# Error Normalized Distance Simulations
A first study of the ideal error normalized distance distributions was done in 'crossmatch_analysis.ipynb'

In this notebook, we include all code and plots to study how error normalized distributions should behave, and change under varying distances and covariances

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Import own modules
import sys
# ESA pc
sys.path.append('/home/rkievit/masterproject/masterproject/src')
# Home PC
sys.path.append('F:\\OneDrive - Telecom Business Transformers BV\\School\\Uni\\MSc Year 2\\Master Research Project\\Project\\masterproject\\src')
# Uni PC
sys.path.append('/net/student36/data2/kievit/Master Project/masterproject/src')
sys.path.append('/home/rens/projects/mp_dir/masterproject/src/')

from plotting_functions import set_styles
from astrometry_equations import rayleigh, compute_error_normalized_distance
set_styles()

# Simulate Samples

In [ ]:
# MOVE SOMEWHERE ELSE
def simulate_error_normalized_distance(dist_mean1, dist_mean2, unc1, unc2, num_samples, method='full', coordinates='cartesian', units='rad'):
    """Simulates num_samples pairs of data points and computes the error normalized distances between them

    TODO: Expand documentation, expand flexibility of code"""

    if len(dist_mean1) == 2:
        # Should check here if we have just 2 vals, or an array of vals
        pass

    if len(unc1) == 3:
        # Check here if all points have same spread, or if it varies
        sigma_x1, sigma_y1, sigma_xy1 = unc1
        sigma_x2, sigma_y2, sigma_xy2 = unc2

        cov1 = [[sigma_x1, sigma_xy1], [sigma_xy1, sigma_y1]]
        cov2 = [[sigma_x2, sigma_xy2], [sigma_xy2, sigma_y2]]

    pos1 = np.random.multivariate_normal(dist_mean1, cov1, num_samples)
    pos2 = np.random.multivariate_normal(dist_mean2, cov2, num_samples)

    unc1 = np.array([np.full(num_samples, sigma_x1), np.full(num_samples, sigma_y1), np.full(num_samples, sigma_xy1)]).T
    unc2 = np.array([np.full(num_samples, sigma_x2), np.full(num_samples, sigma_y2), np.full(num_samples, sigma_xy2)]).T

    en_dist = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method=method, coordinates=coordinates, unit=units)

    return en_dist

In [ ]:
num_samples = 500000
dist_mean1 = [0,0]
dist_mean2 = [0.,0]
unc1 = [8, 6, 0.4]
unc2 = [3, 4, -0.1]

error_normalized_distances = simulate_error_normalized_distance(dist_mean1, dist_mean2, unc1, unc2, num_samples, method='full')

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot()

ax.hist(error_normalized_distances, bins=50, histtype='step', density=True, label='Error Normalized Distance')
x = np.linspace(0,6,50)
ax.plot(x, rayleigh(x,1), label=r'Rayleigh $\sigma = 1$')

ax.set_xlabel('Simulated Uncertainty Normalized Distance')
ax.set_ylabel('Normalized Density')
ax.set_title('Simulated Error Normalized Distance Distribution')

plt.legend()
plt.show()

# Simulate Pairs

In [ ]:
# CODE FROM A.G.A. Brown
def error_ellipses(mu, covmat, sigma_levels, **kwargs):
    """
    Given a covariance matrix for a 2D Normal distribution calculate the uncertainty-ellipses and return
    matplotlib patches for plotting them.
    Parameters
    ----------
    mu : float array
        Mean of Normal distribution (2-vector)
    covmat : float array
        Covariance matrix stored as [sigma_x^2, sigma_y^2, sigma_xy]
    sigma_levels : float or 1-D array
        Equivalent n-sigma levels to draw
    Returns
    -------
    patches : list of matplotlib.patches.Ellipse
        List of matplotlib.patches.Ellipse objects
    Other parameters
    ----------------
    **kwargs :
        Extra arguments for matplotlib.patches.Ellipse
    """
    import matplotlib as mpl
    from scipy.special import erf

    sigmaLevels2D = -2.0 * np.log(
        1.0 - erf(np.array([sigma_levels]).flatten() / np.sqrt(2.0))
    )

    eigvalmax = 0.5 * (
        covmat[0]
        + covmat[1]
        + np.sqrt((covmat[0] - covmat[1]) ** 2 + 4 * covmat[2] ** 2)
    )
    eigvalmin = 0.5 * (
        covmat[0]
        + covmat[1]
        - np.sqrt((covmat[0] - covmat[1]) ** 2 + 4 * covmat[2] ** 2)
    )
    angle = np.arctan2((covmat[0] - eigvalmax), -covmat[2]) / np.pi * 180
    errEllipses = []
    for csqr in sigmaLevels2D:
        errEllipses.append(
            mpl.patches.Ellipse(
                mu,
                2 * np.sqrt(csqr * eigvalmax),
                2 * np.sqrt(csqr * eigvalmin),
                angle=angle,
                **kwargs
            )
        )

    return errEllipses

# MOVE THIS TO ASTROMETRY EQUATIONS


In [ ]:
# Simulate two points with the same mean, at some distance from each other.
mean1 = np.array([0,0])
mean2 = mean1

unc1 = np.array([16, 25, 10]) # sigma_x^2, sigma_y^2, sigma_xy
unc2 = np.array([4, 36, 4])

cov1 = [[unc1[0], unc1[2]], [unc1[2], unc1[1]]]
cov2 = [[unc2[0], unc2[2]], [unc2[2], unc2[1]]]

# sigma's not squared?
#cov1 = [[np.sqrt(unc1[0]), unc1[2]], [unc1[2], np.sqrt(unc1[1])]]
#cov2 = [[np.sqrt(unc2[0]), unc2[2]], [unc2[2], np.sqrt(unc2[1])]]

# Sample two points from this distribution
pos1 = np.random.multivariate_normal(mean1, cov1, 1)[0]
pos2 = np.random.multivariate_normal(mean2, cov2, 1)[0]

# Compute distance and error normalized distances
distance = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='none', coordinates='cartesian')
D_simple = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='simple', coordinates='cartesian')
D_directional = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='directional', coordinates='cartesian')
D_full = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='full', coordinates='cartesian')



# Draw error ellipses individually using agapylib
ellipse_kwargs = {
    'fc': None,
    'fill': False
}
ellipse1 = error_ellipses(pos1, unc1, 1, **ellipse_kwargs, ec='C0')
ellipse2 = error_ellipses(pos2, unc2, 1, **ellipse_kwargs, ec='C1')
ellipse_combined = error_ellipses(pos1, unc1+unc2, 1, **ellipse_kwargs, ec='red', label=rf'Combined Uncertainty [1, {D_simple:.2f}, {D_directional:.2f}, {D_full:.2f}]$\sigma$')
ellipse_combined_D = error_ellipses(pos1, unc1+unc2, [D_simple, D_directional, D_full], **ellipse_kwargs, ec='red', ls='--')#$, label=rf'Combined Uncertainty {D_full:.2f}$\sigma$')

# Plot everything
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot()

ax.scatter(mean1[0], mean1[1], c='C0', marker='X')
ax.scatter(mean2[0], mean2[1], c='C1', marker='X')

ax.scatter(pos1[0], pos1[1], c='C0')
ax.scatter(pos2[0], pos2[1], c='C1')
ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], c='black', ls='--', label=f'Distance: {distance:.2f}')

# Plot ellipses
ax.add_patch(ellipse1[0])
ax.add_patch(ellipse2[0])

ax.add_patch(ellipse_combined[0])
for i in range(len(ellipse_combined_D)):
    ax.add_patch(ellipse_combined_D[i])

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'Simulated Data Points')

plt.legend()
plt.show()

In [ ]:
# Test calculations. See notebook
# Remember, unc should be inserted as [sigma_x^2, sigma_y^2, sigma_xy]
# 1.
pos1, pos2 = np.array([2,0]), np.array([0,1])
unc1, unc2 = np.array([1,4,0]), np.array([4, 1, 0])
print(f"r = {compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='full', coordinates='cartesian')}")
print('')

# Compute distance and error normalized distances
distance = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='none', coordinates='cartesian')
D_simple = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='simple', coordinates='cartesian')
D_directional = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='directional', coordinates='cartesian')
D_full = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='full', coordinates='cartesian')



# Draw error ellipses individually using agapylib
ellipse_kwargs = {
    'fc': None,
    'fill': False
}
ellipse1 = error_ellipses(pos1, unc1, 1, **ellipse_kwargs, ec='C0')
ellipse2 = error_ellipses(pos2, unc2, 1, **ellipse_kwargs, ec='C1')
ellipse_combined = error_ellipses(pos1, unc1+unc2, 1, **ellipse_kwargs, ec='red', label=rf'Combined Uncertainty [1, {D_simple:.2f}, {D_directional:.2f}, {D_full:.2f}]$\sigma$')
ellipse_combined_D = error_ellipses(pos1, unc1+unc2, [D_simple, D_directional, D_full], **ellipse_kwargs, ec='red', ls='--')#$, label=rf'Combined Uncertainty {D_full:.2f}$\sigma$')
ellipse_extra = error_ellipses(pos1, unc1+unc2, 0.707, **ellipse_kwargs, ec='red', ls='--')#$, label=rf'Combined Uncertainty {D_full:.2f}$\sigma$')

# Plot everything
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot()

ax.scatter(mean1[0], mean1[1], c='C0', marker='X')
ax.scatter(mean2[0], mean2[1], c='C1', marker='X')

ax.scatter(pos1[0], pos1[1], c='C0')
ax.scatter(pos2[0], pos2[1], c='C1')
ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], c='black', ls='--', label=f'Distance: {distance:.2f}')

# Plot ellipses
ax.add_patch(ellipse1[0])
ax.add_patch(ellipse2[0])
ax.add_patch(ellipse_extra[0])

ax.add_patch(ellipse_combined[0])
for i in range(len(ellipse_combined_D)):
    ax.add_patch(ellipse_combined_D[i])

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'Simulated Data Points')

plt.legend()
plt.savefig('temp')
plt.show()




In [ ]:
#2.
mean1 = np.array([0,0])
mean2 = np.array([0,0])
pos1, pos2 = np.array([4,7]), np.array([6,8])
unc1, unc2 = np.array([4,4,0.75]), np.array([4,16,-0.2])
print(f"r = {compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='full', coordinates='cartesian')}")

# Compute distance and error normalized distances
distance = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='none', coordinates='cartesian')
D_simple = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='simple', coordinates='cartesian')
D_directional = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='directional', coordinates='cartesian')
D_full = compute_error_normalized_distance(pos1, pos2, unc1, unc2, method='full', coordinates='cartesian')



# Draw error ellipses individually using agapylib
ellipse_kwargs = {
    'fc': None,
    'fill': False
}
ellipse1 = error_ellipses(pos1, unc1, 1, **ellipse_kwargs, ec='C0')
ellipse2 = error_ellipses(pos2, unc2, 1, **ellipse_kwargs, ec='C1')
ellipse_combined = error_ellipses(pos1, unc1+unc2, 1, **ellipse_kwargs, ec='red', label=rf'Combined Uncertainty [1, {D_simple:.2f}, {D_directional:.2f}, {D_full:.2f}]$\sigma$')
ellipse_combined_D = error_ellipses(pos1, unc1+unc2, [D_simple, D_directional, D_full], **ellipse_kwargs, ec='red', ls='--')#$, label=rf'Combined Uncertainty {D_full:.2f}$\sigma$')

# Plot everything
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot()

ax.scatter(mean1[0], mean1[1], c='C0', marker='X')
ax.scatter(mean2[0], mean2[1], c='C1', marker='X')

ax.scatter(pos1[0], pos1[1], c='C0')
ax.scatter(pos2[0], pos2[1], c='C1')
ax.plot([pos1[0], pos2[0]], [pos1[1], pos2[1]], c='black', ls='--', label=f'Distance: {distance:.2f}')

# Plot ellipses
ax.add_patch(ellipse1[0])
ax.add_patch(ellipse2[0])

ax.add_patch(ellipse_combined[0])
for i in range(len(ellipse_combined_D)):
    ax.add_patch(ellipse_combined_D[i])

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_title(f'Simulated Data Points')

plt.legend()
plt.savefig('temp')
plt.show()
